In [2]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [33]:
from astroquery.utils.tap.core import Tap
import pandas as pd
from tqdm.auto import tqdm
import time

# ned dataframe already exists
# required column: ned_name  (if yours is z_ned_name, change below)
ned = pd.read_csv('../../Trash/A2199_NED_redshift_reference_code_old.csv')

tap = Tap(url="https://ned.ipac.caltech.edu/tap")

def get_zflag_by_prefname(ned_name, max_retries=3, retry_delay=5):
    """
    Query NED TAP objdir by prefname and return zflag.
    """
    safe_name = str(ned_name).replace("'", "''")
    query = (
        "SELECT prefname, zflag "
        "FROM NEDTAP.objdir "
        f"WHERE prefname = '{safe_name}'"
    )

    for attempt in range(max_retries):
        try:
            job = tap.launch_job(query)
            table = job.get_results()
            if len(table) == 0:
                return ""
            zflag = str(table["zflag"][0]).strip()
            return zflag
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                return ""

zflags = []
is_specz = []

for name in tqdm(ned["ned_name"], total=len(ned), desc="Querying NED zflag"):
    zf = get_zflag_by_prefname(name)
    zflags.append(zf)
    is_specz.append(zf.upper().startswith("S"))  # first char == S

ned["zflag"] = zflags
ned["is_specz"] = is_specz

# spec-z only
ned_specz = ned[ned["is_specz"]].copy()

# check
display(ned[["ned_name", "zflag", "is_specz"]].head())
print(f"Total: {len(ned):,}, spec-z: {ned['is_specz'].sum():,}, non-spec-z: {(~ned['is_specz']).sum():,}")

Querying NED zflag:   0%|          | 0/100 [00:00<?, ?it/s]

,ned_name,zflag,is_specz
0,SDSS J162541.17+392744.8,SLS,True
1,WISEA J162546.18+394328.2,SUN,True
2,WISEA J162554.36+392730.9,SUN,True
3,WISEA J162606.54+392817.5,SLS,True
4,SDSS J162611.25+394726.8,PUN,False


Total: 100, spec-z: 43, non-spec-z: 57


In [36]:
removed_rows = ned[ned['is_specz']==False]

In [38]:
removed_rows[removed_rows['galaxyflag']==1]

,phot_source,p_phtype0,p_radgal,p_objid,p_ra,p_dec,p_petromag_u,p_petromagerr_u,p_petromag_g,p_petromagerr_g,p_petromag_r,p_petromagerr_r,p_petromag_i,p_petromagerr_i,p_petromag_z,p_petromagerr_z,p_modelmag_u,p_modelmagerr_u,p_modelmag_g,p_modelmagerr_g,p_modelmag_r,p_modelmagerr_r,p_modelmag_i,p_modelmagerr_i,p_modelmag_z,p_modelmagerr_z,p_fibermag_u,p_fibermagerr_u,p_fibermag_g,p_fibermagerr_g,p_fibermag_r,p_fibermagerr_r,p_fibermag_i,p_fibermagerr_i,p_fibermag_z,p_fibermagerr_z,p_extinction_u,p_extinction_g,p_extinction_r,p_extinction_i,p_extinction_z,p_petrorad_r,p_petroraderr_r,p_devrad_i,p_devab_i,p_run,p_rerun,p_camcol,p_field,p_efac,p_probpsf,z_mmt_xcr,z_mmt_tfilename,z_dfilename,z_filename,z_mmt_z,z_mmt_zerr,z_mmt_velqual,z_ned_name,z_ned_z,z_ned_zerr,z_sdss_z,z_sdss_zerr,z_desi_z,z_desi_zerr,z_tot_z,z_tot_zerr,z_tot_zsource,member,p_petromag_r_absmag,galaxyflag,ID,p_petromag_r_0,ned_name,redshift,reference_code,zflag,is_specz
5,DR9,9,28.119853,1237659325492428828,246.584417,39.395512,24.83463,2.392063,22.60099,0.316986,20.74707,0.123059,19.81245,0.254201,19.07485,0.195625,24.95799,0.975094,22.64837,0.147909,20.70655,0.041637,19.89619,0.032147,19.29678,0.074732,24.55211,1.041748,22.73371,0.125092,21.13089,0.074462,20.36804,0.113110,19.77267,0.087658,0.050388,0.037075,0.026890,0.020390,0.014457,1.792526,0.297587,0.471583,0.873853,3225,301,3,237,-0.383820,0,-9.0,NN,NN,nn.fits,-9.0,-9.0,N,WHL J162620.3+392343,0.4515,-9.0,-9.0,-9.0,-9.0,-9.0,0.4515,-9.0,ned,N,-22.152252,1,922,20.720180,NaN,NaN,2016SDSSD.C...0000:,,False
17,DR9,9,32.164907,1237659330851963540,246.870314,40.037018,22.14596,0.573706,21.73022,0.141041,21.15689,0.141994,20.72976,0.163394,20.40938,0.499252,23.23952,0.636420,21.54456,0.061648,21.18927,0.059169,21.01530,0.080493,20.98633,0.256170,23.10972,0.574845,21.83168,0.082119,21.46348,0.091902,21.20930,0.109431,21.01313,0.309634,0.050638,0.037259,0.027023,0.020491,0.014528,1.456294,0.368298,0.662412,0.345915,3226,301,5,97,-0.306595,0,-9.0,NN,NN,nn.fits,-9.0,-9.0,N,SDSS J162728.87+400213.2,3.0450,-9.0,-9.0,-9.0,-9.0,-9.0,3.0450,-9.0,ned,N,-30.185108,1,3539,21.129867,SDSS J162728.87+400213.2,NaN,2009ApJS..180...67R,PUN,False
23,DR9,9,19.819306,1237659330315355299,246.960339,39.255890,24.65330,2.898885,22.77737,0.381243,21.44495,0.178704,20.74109,0.180968,20.75730,0.720457,26.73616,0.403938,22.87981,0.213918,21.30293,0.077532,20.76452,0.061531,20.66936,0.265166,25.87136,0.760035,23.38331,0.295826,21.75929,0.111215,21.16589,0.094425,20.89367,0.332224,0.043221,0.031801,0.023065,0.017489,0.012400,1.620759,0.273485,0.281805,0.353313,3226,301,4,101,-0.314346,0,-9.0,NN,NN,nn.fits,-9.0,-9.0,N,WISEA J162750.48+391521.0,4.1250,-9.0,-9.0,-9.0,-9.0,-9.0,4.1250,-9.0,ned,N,-29.401915,1,4503,21.421885,WISEA J162750.48+391521.0,NaN,2009ApJS..180...67R,PUN,False
33,DR9,8,2.509656,1237659326029299955,247.106710,39.561873,24.12958,1.927624,21.95692,0.142733,20.72249,0.072539,19.99291,0.062463,19.83978,0.250106,23.98248,0.921589,21.80128,0.070901,20.58521,0.036921,19.91213,0.032212,19.54010,0.099188,23.06102,0.275589,21.98885,0.078692,20.85473,0.044383,20.22502,0.041164,19.79150,0.125037,0.056343,0.041456,0.030068,0.022799,0.016165,1.450615,0.086242,0.580597,0.892863,3225,301,4,237,-0.132238,0,-9.0,NN,NN,nn.fits,-9.0,-9.0,N,WISEA J162825.61+393342.8,0.5228,-9.0,-9.0,-9.0,-9.0,-9.0,0.5228,-9.0,ned,N,-22.180868,1,6126,20.692422,WISEA J162825.61+393342.8,NaN,2023ApJS..268...17K,PUN,False


In [ ]:
df = pd.read_csv('./A2199_mastercat_within35arcmin.csv')
df_ned = df[df['z_tot_zsource']=='NED']
ned_query_df = pd.read_csv('./NED/A2199_NED_query.csv')
df_mrt = df_ned[['p_objid', 'p_ra', 'p_dec', 'p_petromag_r_0', 'galaxyflag', 'photflag', 'z_tot_z', 'z_tot_zerr', 'z_tot_zsource', 'member']]
# p_ra, p_dec이 정확히 같으므로 단순 merge 사용
df_mrt = df_mrt.merge(
    ned_query_df[["p_ra", "p_dec", "reference_code"]],
    on=["p_ra", "p_dec"],
    how="left"
)

In [17]:
df_mrt['reference_code'].unique()

array(['2016SDSSD.C...0000:', '2017ApJ...842...88S',
       '2023ApJS..267...27Z'], dtype=object)

In [4]:
df[(df['z_tot_z']!=-9) & (df['z_tot_z']<-0.01)]

,phot_source,p_phtype0,p_radgal,p_objid,p_ra,p_dec,p_petromag_u,p_petromagerr_u,p_petromag_g,p_petromagerr_g,p_petromag_r,p_petromagerr_r,p_petromag_i,p_petromagerr_i,p_petromag_z,p_petromagerr_z,p_modelmag_u,p_modelmagerr_u,p_modelmag_g,p_modelmagerr_g,p_modelmag_r,p_modelmagerr_r,p_modelmag_i,p_modelmagerr_i,p_modelmag_z,p_modelmagerr_z,p_fibermag_u,p_fibermagerr_u,p_fibermag_g,p_fibermagerr_g,p_fibermag_r,p_fibermagerr_r,p_fibermag_i,p_fibermagerr_i,p_fibermag_z,p_fibermagerr_z,p_extinction_u,p_extinction_g,p_extinction_r,p_extinction_i,p_extinction_z,p_petrorad_r,p_petroraderr_r,p_devrad_i,p_devab_i,p_run,p_rerun,p_camcol,p_field,p_efac,p_probpsf,z_mmt_xcr,z_mmt_tfilename,z_dfilename,z_filename,z_mmt_z,z_mmt_zerr,z_mmt_velqual,z_ned_name,z_ned_z,z_ned_zerr,z_sdss_z,z_sdss_zerr,z_desi_id,SURVEY,z_desi_z,z_desi_zerr,FLUX_R,FLUX_IVAR_R,z_tot_z,z_tot_zsource,z_tot_zerr,member,p_modelmag_u_0,p_modelmag_g_0,p_modelmag_r_0,p_modelmag_i_0,p_modelmag_z_0,p_petromag_u_0,p_petromag_g_0,p_petromag_r_0,p_petromag_i_0,p_petromag_z_0,grmod,galaxyflag,photflag,p_petromag_r_absmag
5633,DR9,8,5.664663,1237659326029366062,247.063902,39.488518,24.58895,4.566752,20.39934,0.073247,20.24928,0.079994,20.11639,0.165678,20.23253,0.784034,22.37914,0.45344,20.59124,0.041699,20.27944,0.044906,20.11349,0.064955,20.20225,0.298196,23.20305,0.551171,21.56353,0.045747,21.19569,0.060193,21.03389,0.084491,21.28839,0.460558,0.045291,0.033325,0.02417,0.018327,0.012994,3.131977,0.372061,2.692385,0.764167,3225,301,4,238,-0.946403,0,4.74,/Users/hhwang/Research/Work/MMTraw/2019.0429/r...,skysub_a2199a19_1,128.a2199a19_1_762.ms.fits,-0.027213,0.000202,N,NN,-9.0,-9.0,-9.0,-9.0,NN,NaN,-9.0,-9.0,NaN,NaN,-0.027213,MMT,0.000202,N,22.333849,20.557915,20.25527,20.095163,20.189256,24.543659,20.366015,20.22511,20.098063,20.219536,0.302645,1,0,NaN


In [28]:
c = 299792.458  # km/s
z = -0.027213
v_pec = z * c
v_pec

-8158.252159554

In [6]:
temp = pd.read_csv('/Users/jonginpark/sohnix/Research_Archive/2026/Park2026a-A2199-LF/DATA/A2199/z_A2199_Hwang/z_DATA/z_a2199phot21_5DR9_hshwang_35arcmin_cut.csv')

In [8]:
temp[temp['z_mmt_z']!=-9]

,phot_source,p_phtype0,p_radgal,p_objid,p_ra,p_dec,p_petromag_u,p_petromagerr_u,p_petromag_g,p_petromagerr_g,p_petromag_r,p_petromagerr_r,p_petromag_i,p_petromagerr_i,p_petromag_z,p_petromagerr_z,p_modelmag_u,p_modelmagerr_u,p_modelmag_g,p_modelmagerr_g,p_modelmag_r,p_modelmagerr_r,p_modelmag_i,p_modelmagerr_i,p_modelmag_z,p_modelmagerr_z,p_fibermag_u,p_fibermagerr_u,p_fibermag_g,p_fibermagerr_g,p_fibermag_r,p_fibermagerr_r,p_fibermag_i,p_fibermagerr_i,p_fibermag_z,p_fibermagerr_z,p_extinction_u,p_extinction_g,p_extinction_r,p_extinction_i,p_extinction_z,p_petrorad_r,p_petroraderr_r,p_devrad_i,p_devab_i,p_run,p_rerun,p_camcol,p_field,p_efac,p_probpsf,z_mmt_xcr,z_mmt_tfilename,z_dfilename,z_filename,z_mmt_z,z_mmt_zerr,z_mmt_velqual
10,DR9,8,1.167998,1237659326029365310,247.176125,39.562410,29.25447,0.727764,18.01384,0.307635,17.63639,0.562440,17.24438,0.566942,22.77336,28.077120,20.76688,0.125574,18.76500,0.011530,18.14072,0.009920,17.77475,0.010544,17.47156,0.029799,21.19993,0.142913,19.42523,0.058117,18.65709,0.072755,18.27497,0.072106,17.93772,0.033376,0.059821,0.044016,0.031924,0.024207,0.017163,11.393650,0.852278,1.638964,0.771351,3225,301,4,238,-1.020699,0,23.21,/Users/hhwang/Research/Work/MMTraw/2015.0524/r...,skysub_a2199b15_1,257.a2199b15_1_440.ms.fits,0.030835,0.000040,N
11,DR9,9,1.209012,1237659326029365311,247.136564,39.560003,21.37259,0.232200,19.66680,0.048937,18.87924,0.041590,18.47168,0.037276,18.47866,0.114638,21.22178,0.111747,19.54909,0.014068,18.78121,0.010720,18.37703,0.010781,18.30291,0.035419,21.28366,0.121186,19.66268,0.026912,18.91348,0.023422,18.53185,0.021495,18.33780,0.054665,0.057709,0.042462,0.030797,0.023352,0.016557,1.407972,0.020370,0.453203,0.822599,3225,301,4,238,-0.034241,0,14.55,/Users/hhwang/Research/Work/WISEcfa/IndivClust...,skysub_a2199new_1,223.a2199new_1_2.ms.fits,0.033143,0.000080,Q
14,DR9,9,1.511611,1237659326029365322,247.178495,39.568446,22.08870,0.267996,20.62249,0.047416,19.95388,0.042121,19.62124,0.045204,19.51462,0.140945,22.08870,0.267996,20.62249,0.047416,19.95388,0.042121,19.62124,0.045204,19.51462,0.140945,22.08870,0.267996,20.62249,0.047416,19.95388,0.042121,19.62124,0.045204,19.51462,0.140945,0.059599,0.043852,0.031805,0.024117,0.017099,1.321590,0.059607,0.431296,0.717399,3225,301,4,238,0.636457,0,3.44,/Users/hhwang/Research/Work/WISEcfa/IndivClust...,skysub_a2199new_1,299.a2199new_1_3.ms.fits,0.025007,0.000290,N
17,DR9,8,2.007164,1237659326029365304,247.184891,39.575076,20.22726,0.241813,18.00055,0.026038,17.17890,0.022724,16.82979,0.024467,16.67481,0.047184,19.90637,0.070360,17.84119,0.006809,17.03119,0.005299,16.66260,0.005503,16.41393,0.014649,20.89071,0.068539,18.95899,0.011372,18.15273,0.009444,17.79758,0.009788,17.51366,0.022866,0.059173,0.043539,0.031578,0.023945,0.016977,3.666625,0.064761,2.096876,0.703449,3225,301,4,238,-0.973837,0,10.78,/Users/hhwang/Research/Work/MMTraw/2014.0219/r...,skysub_a2199a_1,214.a2199a_1_444.ms.fits,0.030103,0.000059,N
22,DR9,9,1.339375,1237659326029365325,247.129634,39.545080,24.13763,2.741807,21.09160,0.126353,20.45626,0.157772,20.32563,0.226884,19.86480,0.981063,23.56840,0.826221,21.17442,0.047335,20.48552,0.038014,20.32196,0.051576,19.44440,0.103337,22.83102,1.010150,20.97188,0.034032,20.17208,0.036158,19.79194,0.040268,19.65636,0.281798,0.056091,0.041271,0.029933,0.022698,0.016093,2.074614,0.219558,0.958885,0.799876,3225,301,4,238,0.284185,0,6.86,/Users/hhwang/Research/Work/WISEcfa/IndivClust...,skysub_a2199b_1,277.a2199b_1_328.ms.fits,0.026435,0.000137,Q
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11248,DR9,8,29.768544,1237659330852356825,247.657550,39.236872,22.21415,0.800177,22.88157,0.724483,20.72499,0.130017,20.17922,0.124160,19.96438,0.376394,22.75436,0.614721,21.89456,0.117396,20.60483,0.051902,20.35580,0.066954,19.84359,0.148746,23.3

In [1]:
from astropy.table import Table

path = "./A2199_Machine_Readable_Table2.txt"

t = Table.read(
    path,
    format="ascii",
    data_start=0,  # 1-based: 데이터 첫 줄
    names=[
        "ID",
        "p_objid",
        "p_ra",
        "p_dec",
        "rmag",
        "ExtFlag",
        "PhotFlag",
        "z",
        "e_z",
        "r_z",
        "Member",
    ],
)
print(t[:5])

 ID       p_objid          p_ra      p_dec   ...    z     e_z   r_z Member
                           deg        deg    ...                          
--- ------------------- ---------- --------- ... ------- ------ --- ------
  1 1237659325492363706 246.411225 39.486061 ...  0.6151 0.0002   2      0
  2 1237659325492297918 246.414101 39.546131 ...  0.0302 0.0001   2      1
  3 1237659325492298799 246.416353 39.537589 ...  0.7277 0.0001   3      0
  4 1237659325492363724 246.421577 39.462481 ... -0.0001 0.0001   4      0
  5 1237659325492297794 246.424306 39.600097 ...  0.0298 0.0001   2      1


In [2]:
tdf = t.to_pandas()

In [3]:
# galaxyflag, photflag, r_z, member 컬럼 각각에서 고유값 개수를 카운트하고 출력하는 코드
cols_to_count = ['ExtFlag', 'PhotFlag', 'r_z', 'Member']
for col in cols_to_count:
    print(f"\nValue counts for '{col}':")
    print(tdf[col].value_counts())


Value counts for 'ExtFlag':
ExtFlag
1    2569
0     215
Name: count, dtype: int64

Value counts for 'PhotFlag':
PhotFlag
0    2779
1       5
Name: count, dtype: int64

Value counts for 'r_z':
r_z
1    2032
3     384
2     325
4      37
5       4
6       2
Name: count, dtype: int64

Value counts for 'Member':
Member
0    2341
1     443
Name: count, dtype: int64


In [14]:
df[df['z_tot_z']!=-9]

,phot_source,p_phtype0,p_radgal,p_objid,p_ra,p_dec,p_petromag_u,p_petromagerr_u,p_petromag_g,p_petromagerr_g,...,p_petromag_g_0,p_petromag_r_0,p_petromag_i_0,p_petromag_z_0,grmod,photflag,galaxyflag,extended_source_flag,p_petromag_r_absmag,member
13,DR9,9,34.777964,1237659325492363706,246.411225,39.486061,21.53312,0.172459,21.04035,0.047730,...,21.004527,21.321948,20.808909,20.290711,-0.226871,0,0,0,-21.450354,N
22,DR9,9,34.426644,1237659325492297918,246.414101,39.546131,17.78038,0.032196,15.96191,0.003034,...,15.923007,15.114695,14.707195,14.451031,0.825703,0,1,1,-20.292767,Y
28,DR9,9,34.330671,1237659325492298799,246.416353,39.537589,26.86675,1.015380,23.20494,0.890405,...,23.166699,20.898515,19.744099,19.549219,1.871385,0,1,1,-23.745964,N
40,DR9,9,34.491092,1237659325492363724,246.421577,39.462481,21.47327,0.157966,20.73926,0.038700,...,20.704160,20.876823,21.006826,20.327864,-0.103793,0,0,0,NaN,N
46,DR9,9,34.080759,1237659325492297794,246.424306,39.600097,16.74888,0.057319,14.66841,0.026213,...,14.632002,13.791384,13.429787,13.198883,0.833888,0,1,1,-21.590870,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12788,DR9,9,32.042901,1237659326566367807,247.846568,39.491773,21.73433,0.207101,21.47736,0.075603,...,21.443180,21.048770,20.984672,20.281142,0.361920,0,0,0,-22.034513,N
12789,DR9,9,31.870654,1237659326566368053,247.846657,39.569308,24.41383,1.262554,21.76498,0.559675,...,21.729652,20.888107,20.624581,20.689564,0.840975,0,0,0,NaN,N
12807,DR9,9,32.072534,1237659326566368184,247.850165,39.518325,22.15113,0.562744,21.82230,0.180326,...,21.787756,20.491096,19.715042,19.411500,1.229150,0,1,1,-22.445463,N
12821,DR9,9,33.299276,1237655373036650815,247.853945,39.692935,18.91147,0.086576,17.38997,0.011433,...,17.347491,16.543841,16.177518,16.007246,0.792040,0,1,1,-18.733795,N


In [12]:
df[df['member']=='Y']

,phot_source,p_phtype0,p_radgal,p_objid,p_ra,p_dec,p_petromag_u,p_petromagerr_u,p_petromag_g,p_petromagerr_g,...,p_petromag_g_0,p_petromag_r_0,p_petromag_i_0,p_petromag_z_0,grmod,photflag,galaxyflag,extended_source_flag,p_petromag_r_absmag,member
22,DR9,9,34.426644,1237659325492297918,246.414101,39.546131,17.78038,0.032196,15.96191,0.003034,...,15.923007,15.114695,14.707195,14.451031,0.825703,0,1,1,-20.292767,Y
46,DR9,9,34.080759,1237659325492297794,246.424306,39.600097,16.74888,0.057319,14.66841,0.026213,...,14.632002,13.791384,13.429787,13.198883,0.833888,0,1,1,-21.590870,Y
335,DR9,9,31.608806,1237659325492363521,246.496283,39.420054,18.45763,0.095917,17.21968,0.009705,...,17.182539,16.570302,16.247914,16.134287,0.627657,0,1,1,-18.827934,Y
471,DR9,9,29.668437,1237659325492363431,246.517355,39.532315,20.46051,0.215468,18.78143,0.021980,...,18.751983,18.139013,17.793685,17.634098,0.698190,0,1,1,-17.351760,Y
478,DR9,9,29.962470,1237659330315157673,246.518611,39.628910,18.26517,0.094584,16.62638,0.014720,...,16.593577,15.817169,15.452730,15.265399,0.764188,0,1,1,-19.755465,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12325,DR9,9,28.293091,1237659326566302103,247.762284,39.623716,19.69174,0.154149,18.07368,0.018431,...,18.041541,17.218250,16.838365,16.567298,0.855761,0,1,1,-18.356987,Y
12333,DR9,9,28.007946,1237659326566367503,247.763388,39.562195,17.09839,0.162330,15.50219,0.046172,...,15.472537,14.667293,14.254602,13.985417,0.836064,0,1,1,-20.886827,Y
12339,DR9,9,32.940883,1237655373036585211,247.764201,39.838499,16.69637,0.031883,14.87817,0.003862,...,14.828663,13.980534,13.644613,13.402516,0.859800,0,1,1,-21.464320,Y
12350,DR9,9,31.667079,1237655373036585263,247.765700,39.793451,18.10020,0.346966,15.64610,0.117938,...,15.600841,14.942434,14.517279,14.696022,0.693827,0,1,1,-20.391567,Y


In [31]:
df_file = pd.read_csv('A2199_mastercat_intermediate_file0.csv')
df = pd.read_csv('A2199_mastercat_within35arcmin.csv')

In [32]:
# galaxyflag, photflag, r_z, member 컬럼 각각에서 고유값 개수를 카운트하고 출력하는 코드
cols_to_count = ['z_tot_zsource']
for col in cols_to_count:
    print(f"\nValue counts for '{col}':")
    print(df_file[col].value_counts())


Value counts for 'z_tot_zsource':
z_tot_zsource
none    10182
MMT      2032
DESI      384
SDSS      329
NED        44
Name: count, dtype: int64


In [38]:
df = pd.read_csv('A2199_mastercat_within35arcmin.csv')
df = df[df['z_tot_z']!=-9]
# galaxyflag, photflag, r_z, member 컬럼 각각에서 고유값 개수를 카운트하고 출력하는 코드
cols_to_count = ['z_tot_zsource']
for col in cols_to_count:
    print(f"\nValue counts for '{col}':")
    print(df[col].value_counts())


Value counts for 'z_tot_zsource':
z_tot_zsource
MMT     2032
DESI     384
SDSS     325
NED       43
Name: count, dtype: int64


In [36]:
df

,phot_source,p_phtype0,p_radgal,p_objid,p_ra,p_dec,p_petromag_u,p_petromagerr_u,p_petromag_g,p_petromagerr_g,p_petromag_r,p_petromagerr_r,p_petromag_i,p_petromagerr_i,p_petromag_z,p_petromagerr_z,p_modelmag_u,p_modelmagerr_u,p_modelmag_g,p_modelmagerr_g,p_modelmag_r,p_modelmagerr_r,p_modelmag_i,p_modelmagerr_i,p_modelmag_z,p_modelmagerr_z,p_fibermag_u,p_fibermagerr_u,p_fibermag_g,p_fibermagerr_g,p_fibermag_r,p_fibermagerr_r,p_fibermag_i,p_fibermagerr_i,p_fibermag_z,p_fibermagerr_z,p_extinction_u,p_extinction_g,p_extinction_r,p_extinction_i,p_extinction_z,p_petrorad_r,p_petroraderr_r,p_devrad_i,p_devab_i,p_run,p_rerun,p_camcol,p_field,p_efac,p_probpsf,z_mmt_xcr,z_mmt_tfilename,z_dfilename,z_filename,z_mmt_z,z_mmt_zerr,z_mmt_velqual,z_ned_name,z_ned_z,z_ned_zerr,z_sdss_z,z_sdss_zerr,z_desi_id,SURVEY,z_desi_z,z_desi_zerr,FLUX_R,FLUX_IVAR_R,z_tot_z,z_tot_zsource,z_tot_zerr,p_modelmag_u_0,p_modelmag_g_0,p_modelmag_r_0,p_modelmag_i_0,p_modelmag_z_0,p_petromag_u_0,p_petromag_g_0,p_petromag_r_0,p_petromag_i_0,p_petromag_z_0,grmod,photflag,galaxyflag,extended_source_flag,p_petromag_r_absmag,member
13,DR9,9,34.777964,1237659325492363706,246.411225,39.486061,21.53312,0.172459,21.04035,0.047730,21.34793,0.095170,20.82861,0.099543,20.30468,0.253471,21.51204,0.105899,21.02543,0.031422,21.24246,0.048973,20.77954,0.048102,20.52372,0.154779,21.81192,0.110952,21.32640,0.045484,21.53784,0.081282,21.00543,0.085494,20.66616,0.253862,0.048687,0.035823,0.025982,0.019701,0.013969,1.027648,0.094771,0.183463,0.099945,3225,301,3,236,-0.189915,1,-9.0,NN,NN,nn.fits,-9.0,-9.0,N,NN,-9.000000,-9.000000,0.615174,0.000207,NN,NaN,-9.000000,-9.000000,NaN,NaN,0.6151,SDSS,0.0002,21.463353,20.989607,21.216478,20.759839,20.509751,21.484433,21.004527,21.321948,20.808909,20.290711,-0.226871,0,0,0,-21.450354,N
22,DR9,9,34.426644,1237659325492297918,246.414101,39.546131,17.78038,0.032196,15.96191,0.003034,15.14291,0.002223,14.72859,0.002408,14.46620,0.007496,17.83224,0.017446,15.90852,0.002988,15.07213,0.002544,14.66728,0.002587,14.37994,0.004518,19.30458,0.024915,17.37049,0.003513,16.51471,0.002461,16.08678,0.001905,15.80260,0.005404,0.052872,0.038903,0.028215,0.021395,0.015169,6.194714,0.034543,2.545864,0.946602,3225,301,3,235,-1.371800,0,-9.0,NN,NN,nn.fits,-9.0,-9.0,N,WISEA J162539.39+393246.6,0.030210,0.000010,0.030206,0.000010,NN,NaN,-9.000000,-9.000000,NaN,NaN,0.0302,SDSS,0.0001,17.779368,15.869617,15.043915,14.645885,14.364771,17.727508,15.923007,15.114695,14.707195,14.451031,0.825703,0,1,1,-20.292767,Y
28,DR9,9,34.330671,1237659325492298799,246.416353,39.537589,26.86675,1.015380,23.20494,0.890405,20.92625,0.179344,19.76513,0.105081,19.56413,0.371210,26.01390,1.009866,22.76074,0.297785,20.87885,0.085406,19.61023,0.045613,19.03205,0.114832,25.26507,0.854741,23.84291,0.380128,22.07570,0.131341,20.73041,0.066007,20.17650,0.165384,0.051972,0.038241,0.027735,0.021031,0.014911,2.969671,-9.000000,1.564909,0.527045,3225,301,3,235,-1.149448,0,-9.0,NN,NN,nn.fits,-9.0,-9.0,N,WISEA J162539.93+393215.8,0.729400,0.000360,0.729388,0.000409,39633068444617110,main,0.727788,0.000133,3.113775,136.562134,0.7277,DESI,0.0001,25.961928,22.722499,20.851115,19.589199,19.017139,26.814778,23.166699,20.898515,19.744099,19.549219,1.871385,0,1,1,-23.745964,N
40,DR9,9,34.491092,1237659325492363724,246.421577,39.462481,21.47327,0.157966,20.73926,0.038700,20.90228,0.062189,21.02613,0.110773,20.34155,0.262646,21.33651,0.092739,20.71775,0.025341,20.81190,0.034848,20.78541,0.047812,20.68638,0.173066,21.70148,0.147335,21.01451,0.035138,21.16555,0.057641,21.19552,0.096372,20.80222,0.285351,0.047704,0.035100,0.025457,0.019304,0.013686,1.004089,0.086325,0.016368,0.138338,3225,301,3,236,-0.263269,1,-9.0,NN,NN,nn.fits,-9.0,-9.0,N,SDSS J162541.17+392744.8,-0.000143,0.000187,-9.000000,-9.000000,NN,NaN,-9.000000,-9.000000,NaN,NaN,-0.0001,NED,0.0001,21.288806,20.682650,20.786443,20.766106,20.672694,21.425566,20.704160,20.876823,21.006826,20.327864,-0.103793,0,0,0,NaN,N
46,DR9,9,34.080759,1237659325492297794,

In [14]:
# Open a text file and write the header + data manually
with open('temp.txt', 'w') as f:
    # f.write("index TARGET_RA TARGET_DEC\n")
    temp = df.sort_values(by='p_modelmag_r', ascending=True)
    for idx, row in temp.iterrows():
        f.write(f"{row['p_objid']} {row['p_ra']:.6f} {row['p_dec']:.6f}\n")


In [15]:
ned = pd.read_csv('./NED/A2199_NED_query.csv')

In [17]:
ned[ned['p_objid']==1237659325492363724]

,p_objid,p_ra,p_dec,ned_name,reference_code,redshift,redshift_uncertainty,ned_ra,ned_dec,type,separation_arcsec
77,1237659325492363724,246.421577,39.462481,SDSS J162541.17+392744.8,2016SDSSD.C...0000:,-0.000143,0.000187,246.42157,39.462484,*,0.021321
